#### Problem 8

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the correlation by resampling paired rows. Report the seed and number of bootstrap replications.

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/ames_prices.csv")

df["area_transformed"] = np.arcsinh(df["area"])
df["price_transformed"] = np.arcsinh(df["price"])


paired_data = df[["area_transformed", "price_transformed"]].dropna()

original_correlation = paired_data["area_transformed"].corr(
    paired_data["price_transformed"]
)

In [4]:
seed = 100
n_bootstrap = 2000
rng = np.random.default_rng(seed)

bootstrap_correlations = []

for _ in range(n_bootstrap):
    bootstrap_sample = paired_data.sample(
        n=len(paired_data),
        replace=True,
        random_state=rng
    )

    correlation = bootstrap_sample["area_transformed"].corr(
        bootstrap_sample["price_transformed"]
    )

    bootstrap_correlations.append(correlation)

bootstrap_correlations = np.array(bootstrap_correlations)

print(f"Pearson correlation: {original_correlation:.4f}")
print(f"Bootstrap mean correlation: {np.mean(bootstrap_correlations):.4f}")
print(f"Seed: {seed}")
print(f"Bootstrap replications: {n_bootstrap}")


Pearson correlation: 0.7232
Bootstrap mean correlation: 0.7231
Seed: 100
Bootstrap replications: 2000


b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the 1/(N −1) degrees of freedom correction).

In [6]:
bootstrap_se = np.std(bootstrap_correlations, ddof=1)

print(f"Bootstrap SE: {bootstrap_se:.6f}")

Bootstrap SE: 0.010133


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [7]:
critical_values = {
    0.90: 1.645,
    0.95: 1.96,
    0.99: 2.576
}

confidence_intervals = {}

for confidence, critical_value in critical_values.items():
    margin_error = critical_value * bootstrap_se

    lower = original_correlation - margin_error
    upper = original_correlation + margin_error

    confidence_intervals[confidence] = (lower, upper)

    print(f"{confidence:.0%} CI: [{lower:.4f}, {upper:.4f}]")


90% CI: [0.7065, 0.7398]
95% CI: [0.7033, 0.7430]
99% CI: [0.6971, 0.7493]


Use each confidence interval to evaluate $H_0: p = 0$. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret the association without making a causal claim.

In [ ]:
null_correlation = 0

for confidence, (lower, upper) in confidence_intervals.items():
    alpha = 1 - confidence

    if lower <= null_correlation <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"Alpha = {alpha:.2f}: {decision}")


Alpha = 0.10: Reject H0
Alpha = 0.05: Reject H0
Alpha = 0.01: Reject H0


We have a positive correlation, meaning there is a positive relationship in area and price. Since zero does not fall within any of the confidence intervals, we reject the null hypothesis at all three significance levels. This means there is a significant positive relationship between area and price, but it doesn't mean that one causes the other.

e. Briefly explain why area and price must remain paired during resampling.

Because we don't want to wrongly pair prices with different areas, since each price belongs to a specific house and its area.